# HALO — Kaggle training pipeline
**Player recognition + tactical intelligence engine — adapted from Team Transformers' Code Cup 2026 "Number Cruncher" submission**

Targets a single Kaggle **Tesla T4 (16 GB)** session, **6–8 hours total wall-clock time**, checkpoints every **5 epochs**, and a **100-epoch** ceiling on every trained model, ending with a zipped output package and a demo run on a test video.

## What gets trained here vs. reused pretrained

| Component | Status | Why |
|---|---|---|
| YOLOv8/v11 detector (ball/player/goalkeeper/referee) | **Trained** (fine-tuned) | Small public dataset converges in ~2h on one T4 |
| Jersey-CNN (two-digit jersey number reader) | **Trained** (from scratch) | Synthetic data generated in this notebook, no NDA needed to get started |
| C-CNN FIR filter (temporal track smoother) | **Trained** (from scratch) | Learns a general denoising prior from synthetic trajectories, converges in minutes |
| ByteTrack, k-means team colors, proximity/event rules, dynamic ROI | **Not trained** | Classical/algorithmic, runs at inference time |
| SoccerChat VLM (commentary + tactics) | **Pretrained**, optional light LoRA | Full fine-tuning of a 7B video-LLM needs multi-A100; a single T4 runs it in inference/few-shot mode instead |

## Sources this notebook reconciles

- **Team Transformers' Code Cup 2026 deck** — HALO's 3-tier architecture: Perception (YOLO + dynamic ROI) → Processing (jersey OCR + C-CNN FIR filter) → Intelligence (event log + VLM commentary + tactical overlay).
- **`arxiv.org/abs/2412.17637`** (SCBench / CommentarySet) — this is a **benchmark and evaluation dataset** for scoring sports commentary quality (a six-axis SCORES metric), not an architecture to copy. It's used here only as a guide for what "good" generated commentary should look like.
- **`github.com/AntoineBohin/soccer-ai-commentator`** — this is an **orchestrated pipeline**, not one trained model: YOLOv8+ByteTrack tracking, a Qwen2.5-VL vision-language model, a Gemma-3 LLM, and Zonos TTS are all used **pretrained**, with only its action-spotting CNN actually trained on SoccerNet. That's the answer to "isme model kese niklega" — there isn't a single model that emerges from it; a few small models get trained and the rest are prompted. This notebook follows the same philosophy.
- **`huggingface.co/datasets/SimulaMet/SoccerChat`** — the dataset actually built for soccer VLM commentary and tactical Q&A, with an already-fine-tuned checkpoint (`SimulaMet/SoccerChat-qwen2-vl-7b`) this notebook loads directly.

## Phase budget (fits inside 6–8h with margin)

| Phase | Component | Budget |
|---|---|---|
| 0 | Setup + dataset prep | 20–30 min |
| 1 | YOLO detector fine-tune | ≤2.5 h |
| 2 | Jersey-CNN training | ≤1.5 h |
| 3 | C-CNN FIR filter training | ≤0.5 h |
| 4 | Classical modules (no training) | ~15 min wiring |
| 5 | VLM commentary + tactics (pretrained) | ≤1.5 h |
| 6 | Demo run + zip package | ~20–30 min |
| **Total** | | **~5.5–6.6 h**, safety margin inside the 6–8h target |

In [ ]:
!pip install -q ultralytics==8.3.109 supervision==0.24.0 opencv-python-headless timm einops scikit-learn pyyaml
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils[decord]
!pip install -q roboflow
print('Install complete')

In [ ]:
import os, sys, json, time, random, shutil, math, glob
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs visible: {N_GPUS}')
if torch.cuda.is_available():
    for i in range(N_GPUS):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/halo_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(exist_ok=True)
(OUTPUT_DIR / 'demo').mkdir(exist_ok=True)

CFG = {
    'total_budget_hours': 7.0,          # overall wall-clock ceiling, keep inside the 6-8h target
    'yolo_epochs': 100,
    'yolo_time_budget_hours': 2.5,      # ultralytics native time-based cutoff
    'yolo_batch': 32,                   # yolov8n at 640px comfortably fits a single T4; halve if you pick yolov8s
    'yolo_imgsz': 640,
    'jersey_epochs': 100,
    'jersey_time_budget_hours': 1.5,
    'jersey_batch': 256,
    'ccnn_epochs': 100,
    'ccnn_time_budget_hours': 0.5,
    'ccnn_batch': 128,
    'checkpoint_every': 5,              # epochs between checkpoints, all three trained models
    'vlm_time_budget_hours': 1.5,
    'num_workers': 2,                   # Kaggle CPU notebooks give 4 cores; leave headroom for the dataloader + main process
}

class TimeBudget:
    # Wall-clock guard so a phase can never blow through its allotted hours,
    # even if epochs / patience would otherwise keep going.
    def __init__(self, hours):
        self.limit_s = hours * 3600
        self.t0 = time.time()

    def elapsed(self):
        return time.time() - self.t0

    def remaining(self):
        return max(0.0, self.limit_s - self.elapsed())

    def expired(self):
        return self.elapsed() >= self.limit_s

    def __repr__(self):
        return f'{self.elapsed()/60:.1f} min elapsed / {self.limit_s/60:.1f} min budget'

PHASE_LOG = {}   # phase_name -> seconds spent, filled in as we go, printed at the end
def log_phase(name, tb):
    PHASE_LOG[name] = tb.elapsed()
    print(f'[{name}] finished in {tb.elapsed()/60:.1f} min (budget {tb.limit_s/60:.1f} min)')

## Dataset reality check (please read before running)

Neither of the two originally-suggested datasets contains frame-level visual data this pipeline needs:

- **`shots_dataset.csv`** (Understat) is shot-event tabular data — X/Y shot location, xG, player name, minute, situation — for several leagues, 2014–2022. No video, no jersey crops, no touch/pass event stream.
- **`fifa-worldcup-2018-dataset`** is 32 rows of team-level group-stage fixtures and pairwise head-to-head history. No frame-level or vision data either.

Both are kept below as **optional auxiliary tabular context** for the tactical-report generator (e.g. team-strength priors) — never for training the vision models. The three components that actually get trained use purpose-built data instead:

| Trained component | Data used | Why |
|---|---|---|
| YOLO detector | Roboflow `football-players-detection-3zvbc` (663 images, CC BY 4.0; classes ball/goalkeeper/player/referee) | Small enough to converge in ~2h on one T4, exact classes HALO needs |
| Jersey-CNN | Synthetic jersey-digit generator (built below, no download) | Guarantees a working, converging dataset without a SoccerNet NDA wait; swap in the real SoccerNet Jersey Number Recognition set later for higher accuracy |
| C-CNN FIR filter | Synthetic noisy trajectories (generated below) | The filter learns a general denoising prior, not sport-specific patterns — synthetic data is enough |

If you later get the free SoccerNet NDA signed, point `JERSEY_DATA_DIR` at the real tracklets instead of the synthetic set — everything downstream (model, training loop, checkpoints) is unchanged.

In [ ]:
# Roboflow hosts the standard small detection set used across most public
# football-YOLO tutorials: ball / goalkeeper / player / referee, ~663 images.
# Get a free API key at https://app.roboflow.com/ -> Settings -> API Keys,
# then add it as a Kaggle secret or paste it below.
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'YOUR_API_KEY_HERE')
YOLO_DATA_DIR = Path('/kaggle/working/data/football-players-detection')

def download_yolo_dataset():
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
        version = project.version(1)
        dataset = version.download('yolov8', location=str(YOLO_DATA_DIR))
        return Path(dataset.location)
    except Exception as e:
        print(f'Roboflow download failed ({e}).')
        print('Fallback: add the dataset manually via Kaggle "Add Data" -> search')
        print('"football players detection" and point YOLO_DATA_DIR at it, or upload')
        print('a zip of any YOLO-format player/ball/referee dataset with this layout:')
        print('  train/images, train/labels, valid/images, valid/labels, data.yaml')
        return YOLO_DATA_DIR

YOLO_DATA_DIR = download_yolo_dataset()
print('YOLO dataset at:', YOLO_DATA_DIR)

In [ ]:
import yaml

DATA_YAML = YOLO_DATA_DIR / 'data.yaml'
if DATA_YAML.exists():
    print(DATA_YAML.read_text())
else:
    # Write one manually if the auto-download didn't already produce it
    yaml_text = f"""train: {YOLO_DATA_DIR}/train/images
val: {YOLO_DATA_DIR}/valid/images
test: {YOLO_DATA_DIR}/test/images
nc: 4
names: ['ball', 'goalkeeper', 'player', 'referee']
"""
    DATA_YAML.write_text(yaml_text)
    print('Wrote a default data.yaml - check the paths match your uploaded dataset layout.')
    print(yaml_text)

data_cfg = yaml.safe_load(DATA_YAML.read_text())
CLASS_NAMES = data_cfg.get('names', ['ball', 'goalkeeper', 'player', 'referee'])
print('Classes:', CLASS_NAMES)

In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import colorsys

JERSEY_IMG_SIZE = 64
JERSEY_DATA_DIR = Path('/kaggle/working/data/jersey_synth')

def _random_jersey_color():
    # keep colors reasonably saturated so digits stay legible, like real kits
    h = random.random()
    s = random.uniform(0.5, 1.0)
    v = random.uniform(0.4, 0.95)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (int(r * 255), int(g * 255), int(b * 255))

def _digit_color(bg_color):
    # pick a contrasting digit color (white or near-black) based on background luminance
    lum = 0.299 * bg_color[0] + 0.587 * bg_color[1] + 0.114 * bg_color[2]
    return (20, 20, 20) if lum > 150 else (245, 245, 245)

def _load_font(size):
    candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
    ]
    for p in candidates:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def _random_affine_shear(img, size, strength=0.15):
    # light shear as a cheap stand-in for camera-angle distortion (no linear-system solve needed)
    shx = random.uniform(-strength, strength)
    shy = random.uniform(-strength, strength)
    cx, cy = size / 2, size / 2
    a, b, c = 1, shx, -shx * cy
    d, e, f = shy, 1, -shy * cx
    fill = img.getpixel((0, 0))
    return img.transform((size, size), Image.AFFINE, (a, b, c, d, e, f), resample=Image.BICUBIC, fillcolor=fill)

def _render_one(number):
    # number: -1 for "no visible number", else 0-99
    size = JERSEY_IMG_SIZE * 3   # render big, downsample later for anti-aliasing
    bg = _random_jersey_color()
    img = Image.new('RGB', (size, size), bg)
    draw = ImageDraw.Draw(img)

    if number >= 0:
        text = str(number)
        font_size = int(size * random.uniform(0.55, 0.78))
        font = _load_font(font_size)
        color = _digit_color(bg)
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        tx = (size - tw) / 2 - bbox[0] + random.randint(-8, 8)
        ty = (size - th) / 2 - bbox[1] + random.randint(-8, 8)
        draw.text((tx, ty), text, font=font, fill=color)

    angle = random.uniform(-18, 18)
    img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=bg)
    img = _random_affine_shear(img, size)

    # occlusion: a teammate's arm / motion blur strip crossing the digit
    if random.random() < 0.25:
        draw2 = ImageDraw.Draw(img)
        ox = random.randint(0, size)
        ow = random.randint(size // 6, size // 3)
        draw2.rectangle([ox, 0, ox + ow, size], fill=bg)

    img = img.resize((JERSEY_IMG_SIZE, JERSEY_IMG_SIZE), Image.BICUBIC)

    # brightness / contrast jitter + motion blur to mimic broadcast/amateur footage
    arr = np.array(img).astype(np.float32)
    arr *= random.uniform(0.7, 1.3)
    arr += random.uniform(-15, 15)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)
    if random.random() < 0.4:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.6)))

    return img

def generate_synthetic_jersey_dataset(root, n_train=12000, n_val=2000, blank_prob=0.15):
    root = Path(root)
    manifest = {'train': [], 'val': []}
    for split, n in [('train', n_train), ('val', n_val)]:
        split_dir = root / split
        split_dir.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            has_number = random.random() > blank_prob
            number = random.randint(0, 99) if has_number else -1
            img = _render_one(number)
            fname = f'{split}_{i:06d}.png'
            img.save(split_dir / fname)
            manifest[split].append({'file': fname, 'number': number})
        print(f'{split}: generated {n} synthetic jersey crops')
    (root / 'manifest.json').write_text(json.dumps(manifest))
    return root

# ~14k small 64x64 crops render in a couple of minutes on the Kaggle CPU
JERSEY_DATA_DIR = generate_synthetic_jersey_dataset(JERSEY_DATA_DIR, n_train=12000, n_val=2000)

In [ ]:
# Optional: load the two originally-suggested datasets purely as auxiliary
# tactical-report context (team strength priors, historical xG tendencies).
# They are NOT used to train any vision model - see the note above.
import pandas as pd

UNDERSTAT_CSV = Path('/kaggle/input/understat-shots/shots_dataset.csv')     # adjust to your Kaggle "Add Data" mount path
FIFA_WC2018_GLOB = '/kaggle/input/fifa-worldcup-2018-dataset/*.csv'         # adjust filename after adding the dataset

def load_auxiliary_context():
    ctx = {}
    if UNDERSTAT_CSV.exists():
        shots = pd.read_csv(UNDERSTAT_CSV)
        # league-average xG per situation, useful as a prior when the tactical
        # LLM describes how "clinical" a team's chances were
        ctx['xg_by_situation'] = shots.groupby('situation')['xG'].mean().to_dict() if 'situation' in shots.columns else {}
        print('Loaded Understat shots for auxiliary xG priors:', len(shots), 'rows')
    else:
        print('Understat CSV not found at', UNDERSTAT_CSV, '- skipping (optional).')

    fifa_files = glob.glob(FIFA_WC2018_GLOB)
    if fifa_files:
        fifa = pd.read_csv(fifa_files[0])
        ctx['fifa_wc2018_teams'] = fifa.to_dict(orient='records')
        print('Loaded FIFA WC 2018 team/fixture metadata:', len(fifa), 'rows')
    else:
        print('FIFA WC 2018 CSV not found - skipping (optional).')
    return ctx

AUX_CONTEXT = load_auxiliary_context()

### Phase 1 — YOLO detector (trained)

Fine-tunes `yolov8n` on ball/goalkeeper/player/referee. The nano model at 640px/batch 32 comfortably fits a single T4 inside the ~2.5h slot; step up to `yolov8s.pt` only if earlier phases finish early. Ultralytics' native `time=` argument gives a hard wall-clock ceiling in addition to `epochs=100`, and `save_period=5` writes a checkpoint every 5 epochs into `runs/detect/halo_yolo/weights/`.

In [ ]:
from ultralytics import YOLO

yolo_tb = TimeBudget(CFG['yolo_time_budget_hours'])

yolo_model = YOLO('yolov8n.pt')  # swap to yolov8s.pt for extra accuracy if you have time to spare

yolo_results = yolo_model.train(
    data=str(DATA_YAML),
    epochs=CFG['yolo_epochs'],
    time=CFG['yolo_time_budget_hours'],        # hard wall-clock cutoff, independent of epoch count
    batch=CFG['yolo_batch'],
    imgsz=CFG['yolo_imgsz'],
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=20,                                # stop early if val mAP plateaus
    save_period=CFG['checkpoint_every'],        # checkpoint every 5 epochs
    project=str(OUTPUT_DIR / 'yolo'),
    name='halo_yolo',
    amp=True,
    cache=True,
    workers=CFG['num_workers'],
    seed=SEED,
    verbose=True,
)

log_phase('yolo_detector', yolo_tb)
YOLO_BEST = OUTPUT_DIR / 'yolo' / 'halo_yolo' / 'weights' / 'best.pt'
print('Best YOLO weights:', YOLO_BEST, '| exists:', YOLO_BEST.exists())

In [ ]:
if YOLO_BEST.exists():
    val_model = YOLO(str(YOLO_BEST))
    metrics = val_model.val(data=str(DATA_YAML), imgsz=CFG['yolo_imgsz'])
    print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)
else:
    print('Training did not produce best.pt yet - check the logs above.')

### Phase 2 — Jersey-CNN (trained)

A small spatial-transformer + CNN trunk with two classification heads (tens digit, units digit) plus a "number visible" flag, following the two-digit-independent formulation used in the jersey-number-recognition literature (Gerke et al.). Trained on the synthetic set generated above; swap `JERSEY_DATA_DIR` for real SoccerNet tracklets later without changing the model.

In [ ]:
import torchvision.transforms as T

class JerseyDataset(Dataset):
    def __init__(self, root, split, img_size=JERSEY_IMG_SIZE):
        self.root = Path(root) / split
        manifest = json.loads((Path(root) / 'manifest.json').read_text())[split]
        self.items = manifest
        self.tf = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(self.root / item['file']).convert('RGB')
        img = self.tf(img)
        number = item['number']
        visible = 1 if number >= 0 else 0
        tens = number // 10 if number >= 0 else 10   # class 10 = "blank / not visible"
        units = number % 10 if number >= 0 else 10
        return img, torch.tensor(visible, dtype=torch.long), torch.tensor(tens, dtype=torch.long), torch.tensor(units, dtype=torch.long)

In [ ]:
class SpatialTransformer(nn.Module):
    # Learns a small affine correction so off-angle / tilted jersey crops get
    # roughly front-facing before the classifier heads see them.
    def __init__(self, in_ch=3):
        super().__init__()
        self.loc = nn.Sequential(
            nn.Conv2d(in_ch, 16, 7, padding=3), nn.MaxPool2d(2), nn.ReLU(True),
            nn.Conv2d(16, 32, 5, padding=2), nn.MaxPool2d(2), nn.ReLU(True),
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * (JERSEY_IMG_SIZE // 4) ** 2, 64), nn.ReLU(True),
            nn.Linear(64, 6),
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        theta = self.loc(x)
        theta = theta.view(theta.size(0), -1)
        theta = self.fc(theta).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False)


class JerseyCNN(nn.Module):
    def __init__(self, num_digit_classes=11):  # 0-9 + "blank"
        super().__init__()
        self.stn = SpatialTransformer()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.trunk = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.head_visible = nn.Linear(128, 2)
        self.head_tens = nn.Linear(128, num_digit_classes)
        self.head_units = nn.Linear(128, num_digit_classes)

    def forward(self, x):
        x = self.stn(x)
        feat = self.trunk(self.backbone(x))
        return self.head_visible(feat), self.head_tens(feat), self.head_units(feat)

In [ ]:
train_ds = JerseyDataset(JERSEY_DATA_DIR, 'train')
val_ds = JerseyDataset(JERSEY_DATA_DIR, 'val')
train_dl = DataLoader(train_ds, batch_size=CFG['jersey_batch'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=CFG['jersey_batch'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

jersey_model = JerseyCNN().to(DEVICE)
opt = torch.optim.AdamW(jersey_model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['jersey_epochs'])
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
ce = nn.CrossEntropyLoss()

jersey_tb = TimeBudget(CFG['jersey_time_budget_hours'])
best_val_acc = 0.0
ckpt_dir = OUTPUT_DIR / 'checkpoints' / 'jersey_cnn'
ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, CFG['jersey_epochs'] + 1):
    jersey_model.train()
    running_loss = 0.0
    for imgs, vis, tens, units in train_dl:
        imgs, vis, tens, units = imgs.to(DEVICE), vis.to(DEVICE), tens.to(DEVICE), units.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            p_vis, p_tens, p_units = jersey_model(imgs)
            loss = ce(p_vis, vis) + ce(p_tens, tens) + ce(p_units, units)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item()
    sched.step()

    jersey_model.eval()
    correct_full, total = 0, 0
    with torch.no_grad():
        for imgs, vis, tens, units in val_dl:
            imgs, vis, tens, units = imgs.to(DEVICE), vis.to(DEVICE), tens.to(DEVICE), units.to(DEVICE)
            p_vis, p_tens, p_units = jersey_model(imgs)
            pred_full = (p_vis.argmax(1) == vis) & (p_tens.argmax(1) == tens) & (p_units.argmax(1) == units)
            correct_full += pred_full.sum().item()
            total += imgs.size(0)
    val_acc = correct_full / max(1, total)
    print(f'epoch {epoch:3d} | train_loss {running_loss/len(train_dl):.4f} | val exact-match acc {val_acc:.4f} | {jersey_tb}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(jersey_model.state_dict(), ckpt_dir / 'best.pt')

    if epoch % CFG['checkpoint_every'] == 0:
        torch.save({'epoch': epoch, 'model': jersey_model.state_dict(), 'opt': opt.state_dict(), 'val_acc': val_acc},
                   ckpt_dir / f'epoch_{epoch:03d}.pt')

    if jersey_tb.expired():
        print(f'Time budget for Jersey-CNN reached at epoch {epoch}, stopping early.')
        break

log_phase('jersey_cnn', jersey_tb)
print('Best Jersey-CNN val exact-match accuracy:', best_val_acc)

In [ ]:
jersey_model.load_state_dict(torch.load(ckpt_dir / 'best.pt', map_location=DEVICE))
jersey_model.eval()

sample_imgs, sample_vis, sample_tens, sample_units = next(iter(val_dl))
with torch.no_grad():
    p_vis, p_tens, p_units = jersey_model(sample_imgs[:12].to(DEVICE))

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, ax in enumerate(axes.flat):
    img = sample_imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(img)
    pred_num = 'blank' if p_vis[i].argmax().item() == 0 else f'{p_tens[i].argmax().item()}{p_units[i].argmax().item()}'
    true_num = 'blank' if sample_vis[i].item() == 0 else f'{sample_tens[i].item()}{sample_units[i].item()}'
    ax.set_title(f'pred {pred_num} / true {true_num}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'logs' / 'jersey_cnn_samples.png', dpi=100)
plt.show()

### Phase 3 — C-CNN temporal FIR filter (trained)

This is the component the team's slides label the "C-CNN FIR filter": a learned temporal convolution stack (a small TCN) that takes a noisy window of `(x, y, confidence)` per track and outputs the smoothed `(x, y)` path — replacing hand-tuned FIR/Savitzky-Golay coefficients with a trained denoiser. Trained entirely on synthetic trajectories, so it needs no match footage and converges in minutes.

In [ ]:
WINDOW = 32   # frames of temporal context per filtering window

def _simulate_clean_trajectory(length):
    # smooth random walk with momentum, standing in for a real player/ball path
    t = np.linspace(0, 1, length)
    freq1, freq2 = np.random.uniform(0.5, 2, 2)
    x = 0.5 + 0.3 * np.sin(2 * np.pi * freq1 * t + np.random.uniform(0, 2 * np.pi))
    y = 0.5 + 0.3 * np.sin(2 * np.pi * freq2 * t + np.random.uniform(0, 2 * np.pi)) * np.cos(t * np.pi)
    return np.stack([x, y], axis=1)

def _corrupt_trajectory(clean):
    noisy = clean.copy()
    noisy += np.random.normal(0, 0.02, noisy.shape)                # detector jitter
    if random.random() < 0.5:                                       # occasional missed-detection gap
        gap_start = random.randint(0, len(noisy) - 6)
        gap_len = random.randint(2, 5)
        noisy[gap_start:gap_start + gap_len] = noisy[max(0, gap_start - 1)]
    conf = np.clip(1 - np.abs(np.random.normal(0, 0.15, len(noisy))), 0.05, 1.0)
    return noisy, conf

class TrajectoryDataset(Dataset):
    def __init__(self, n_samples=20000, window=WINDOW):
        self.window = window
        self.samples = []
        for _ in range(n_samples):
            clean = _simulate_clean_trajectory(window)
            noisy, conf = _corrupt_trajectory(clean)
            self.samples.append((noisy, conf, clean))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        noisy, conf, clean = self.samples[idx]
        inp = np.concatenate([noisy, conf[:, None]], axis=1).astype(np.float32)    # (window, 3)
        target = clean.astype(np.float32)                                          # (window, 2)
        return torch.from_numpy(inp).transpose(0, 1), torch.from_numpy(target).transpose(0, 1)  # (C, T)

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.BatchNorm1d(out_ch)
        self.norm2 = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU(True)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        h = self.act(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act(h + self.skip(x))


class CCNNFilter(nn.Module):
    # Learned FIR-style temporal smoother: stacked dilated 1D convolutions
    # give each output frame a wide receptive field over past+future context.
    def __init__(self, in_ch=3, hidden=32, out_ch=2, n_blocks=4):
        super().__init__()
        blocks = []
        ch = in_ch
        for i in range(n_blocks):
            blocks.append(TemporalBlock(ch, hidden, kernel_size=5, dilation=2 ** i))
            ch = hidden
        self.blocks = nn.Sequential(*blocks)
        self.out = nn.Conv1d(hidden, out_ch, 1)

    def forward(self, x):        # x: (B, 3, T) -> (B, 2, T)
        return self.out(self.blocks(x))

In [ ]:
traj_train = TrajectoryDataset(n_samples=16000)
traj_val = TrajectoryDataset(n_samples=2000)
traj_train_dl = DataLoader(traj_train, batch_size=CFG['ccnn_batch'], shuffle=True, num_workers=CFG['num_workers'])
traj_val_dl = DataLoader(traj_val, batch_size=CFG['ccnn_batch'], shuffle=False, num_workers=CFG['num_workers'])

ccnn = CCNNFilter().to(DEVICE)
opt_c = torch.optim.AdamW(ccnn.parameters(), lr=1e-3, weight_decay=1e-5)
sched_c = torch.optim.lr_scheduler.CosineAnnealingLR(opt_c, T_max=CFG['ccnn_epochs'])
mse = nn.MSELoss()

ccnn_tb = TimeBudget(CFG['ccnn_time_budget_hours'])
best_val_mse = float('inf')
ckpt_dir_c = OUTPUT_DIR / 'checkpoints' / 'ccnn_filter'
ckpt_dir_c.mkdir(parents=True, exist_ok=True)

for epoch in range(1, CFG['ccnn_epochs'] + 1):
    ccnn.train()
    running = 0.0
    for inp, target in traj_train_dl:
        inp, target = inp.to(DEVICE), target.to(DEVICE)
        opt_c.zero_grad(set_to_none=True)
        pred = ccnn(inp)
        loss = mse(pred, target)
        loss.backward()
        opt_c.step()
        running += loss.item()
    sched_c.step()

    ccnn.eval()
    val_running = 0.0
    with torch.no_grad():
        for inp, target in traj_val_dl:
            inp, target = inp.to(DEVICE), target.to(DEVICE)
            val_running += mse(ccnn(inp), target).item()
    val_mse = val_running / len(traj_val_dl)
    print(f'epoch {epoch:3d} | train_mse {running/len(traj_train_dl):.5f} | val_mse {val_mse:.5f} | {ccnn_tb}')

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save(ccnn.state_dict(), ckpt_dir_c / 'best.pt')

    if epoch % CFG['checkpoint_every'] == 0:
        torch.save({'epoch': epoch, 'model': ccnn.state_dict(), 'val_mse': val_mse}, ckpt_dir_c / f'epoch_{epoch:03d}.pt')

    if ccnn_tb.expired():
        print(f'Time budget for C-CNN filter reached at epoch {epoch}, stopping early.')
        break

log_phase('ccnn_filter', ccnn_tb)
print('Best C-CNN filter val MSE:', best_val_mse)

In [ ]:
ccnn.load_state_dict(torch.load(ckpt_dir_c / 'best.pt', map_location=DEVICE))
ccnn.eval()

sample_inp, sample_target = traj_val[0]
with torch.no_grad():
    smoothed = ccnn(sample_inp.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

fig, ax = plt.subplots(figsize=(6, 6))
noisy_xy = sample_inp[:2].T.numpy()
clean_xy = sample_target.T.numpy()
smooth_xy = smoothed.T.numpy()
ax.plot(*noisy_xy.T, 'o--', alpha=0.4, label='noisy input (raw track)')
ax.plot(*clean_xy.T, '-', linewidth=2, label='ground-truth path')
ax.plot(*smooth_xy.T, '-', linewidth=2, label='C-CNN smoothed output')
ax.legend(); ax.set_title('C-CNN FIR filter: denoising a track')
plt.savefig(OUTPUT_DIR / 'logs' / 'ccnn_filter_demo.png', dpi=100)
plt.show()

### Phase 4 — Classical / rule-based modules (no training)

ByteTrack, k-means team-color clustering, proximity-based event rules, and the dynamic ROI engine are algorithmic, not learned — they run at inference time on top of the trained YOLO + Jersey-CNN + C-CNN outputs.

**What "dynamic field ROI" needs beyond YOLO:**
1. **Scale-adaptive radius** — use the nearby player's detected bbox height as a proxy for camera distance/zoom, so the ROI covers a constant real-world radius (e.g. ~5m) instead of a fixed pixel count that's wrong at every zoom level.
2. **A Kalman filter on the ball position** — so the ROI keeps tracking a predicted location through the few frames YOLO misses the ball (motion blur, occlusion), instead of the ROI collapsing or jumping.
3. **Pitch homography** — a 4-point correspondence (or a calibration model) from pixels to real pitch meters, so "5 meters" actually means 5 meters regardless of camera position; without it, every downstream tactical distance is meaningless.
4. *(Optional, not implemented below)* pitch/grass segmentation to keep the ROI search confined to the field of play, and frame-to-frame registration if the amateur camera pans/zooms.

In [ ]:
import supervision as sv
from sklearn.cluster import KMeans

tracker = sv.ByteTrack()

def assign_team_colors(frame, player_boxes, k=2):
    # Sample the shirt region (top third of each player box) and cluster
    # dominant colors into k teams - no training, just per-frame k-means.
    samples = []
    for (x1, y1, x2, y2) in player_boxes:
        shirt = frame[int(y1):int(y1 + (y2 - y1) * 0.35), int(x1):int(x2)]
        if shirt.size == 0:
            samples.append([0, 0, 0])
            continue
        samples.append(shirt.reshape(-1, 3).mean(axis=0))
    samples = np.array(samples)
    if len(samples) < k:
        return np.zeros(len(samples), dtype=int)
    km = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(samples)
    return km.labels_

In [ ]:
BALL_TOUCH_RADIUS_M = 1.5   # real-world meters, converted to pixels via the homography scale

def detect_touches_and_passes(track_history, ball_history, pitch_scale_m_per_px):
    # track_history: {track_id: [(frame_idx, x_px, y_px), ...]}
    # ball_history:  [(frame_idx, x_px, y_px), ...]
    events = []
    last_touch_track = None
    for frame_idx, bx, by in ball_history:
        closest_id, closest_d = None, float('inf')
        for track_id, positions in track_history.items():
            match = next((p for p in positions if p[0] == frame_idx), None)
            if match is None:
                continue
            d_px = math.hypot(match[1] - bx, match[2] - by)
            d_m = d_px * pitch_scale_m_per_px
            if d_m < BALL_TOUCH_RADIUS_M and d_m < closest_d:
                closest_id, closest_d = track_id, d_m
        if closest_id is not None:
            events.append({'frame': frame_idx, 'type': 'touch', 'track_id': closest_id})
            if last_touch_track is not None and last_touch_track != closest_id:
                events.append({'frame': frame_idx, 'type': 'pass', 'from': last_touch_track, 'to': closest_id})
            last_touch_track = closest_id
    return events

In [ ]:
class BallKalmanFilter:
    # Constant-velocity Kalman filter so the ROI keeps tracking the ball's
    # predicted position through brief occlusions / missed detections.
    def __init__(self):
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.transitionMatrix = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.float32)
        self.kf.measurementMatrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
        self.initialized = False

    def update(self, measurement=None):
        if not self.initialized and measurement is not None:
            self.kf.statePre = np.array([measurement[0], measurement[1], 0, 0], dtype=np.float32)
            self.kf.statePost = self.kf.statePre.copy()
            self.initialized = True
        pred = self.kf.predict()
        if measurement is not None:
            self.kf.correct(np.array(measurement, dtype=np.float32))
        return float(pred[0]), float(pred[1])


class DynamicROIEngine:
    # Everything beyond a plain YOLO box: pitch homography for real-world
    # scale, a Kalman filter so the ROI survives brief ball occlusions, and
    # a scale-adaptive radius so the crop covers a constant real-world area
    # regardless of camera zoom/distance.
    def __init__(self, pitch_corners_px=None, pitch_corners_m=None, real_radius_m=5.0, avg_player_height_m=1.8):
        self.kf = BallKalmanFilter()
        self.real_radius_m = real_radius_m
        self.avg_player_height_m = avg_player_height_m
        self.homography = None
        if pitch_corners_px is not None and pitch_corners_m is not None:
            self.homography = cv2.findHomography(np.array(pitch_corners_px, dtype=np.float32),
                                                   np.array(pitch_corners_m, dtype=np.float32))[0]

    def pixel_to_meters(self, px_distance, nearby_player_bbox_height_px):
        # Scale-adaptive: use the nearest player's bbox height as a proxy for
        # camera distance/zoom instead of one global, fixed pixel radius.
        if nearby_player_bbox_height_px <= 0:
            return px_distance
        px_per_meter = nearby_player_bbox_height_px / self.avg_player_height_m
        return px_distance / px_per_meter

    def roi_radius_px(self, nearby_player_bbox_height_px):
        px_per_meter = max(nearby_player_bbox_height_px, 1e-3) / self.avg_player_height_m
        return self.real_radius_m * px_per_meter

    def update(self, ball_detection, nearby_player_bbox_height_px):
        # ball_detection: (x, y) in pixels, or None if YOLO missed the ball this frame
        bx, by = self.kf.update(ball_detection)
        radius_px = self.roi_radius_px(nearby_player_bbox_height_px)
        return {'center': (bx, by), 'radius_px': radius_px}

### Phase 5 — VLM commentary + tactics (pretrained, optional light fine-tune)

A single T4 in a shared ~7h budget cannot realistically LoRA fine-tune a 7B video-LLM end to end — SoccerChat's own recipe uses 4xA100 for that. The practical path: load the already-fine-tuned `SimulaMet/SoccerChat-qwen2-vl-7b` checkpoint in 4-bit and run it in inference/few-shot mode, **grounded with the structured JSON event log** from the earlier phases so the VLM narrates what the tracker/CNNs already measured instead of guessing. A short optional LoRA pass is included further down, gated behind the time budget, for teams that finish earlier phases with hours to spare.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

VLM_MODEL_ID = 'SimulaMet/SoccerChat-qwen2-vl-7b'
VLM_FALLBACK_ID = 'Qwen/Qwen2-VL-2B-Instruct'   # lighter fallback if the 7B model won't fit in 16GB even at 4-bit

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

def load_vlm(model_id):
    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.float16,
    )
    return model, processor

# Note: for exact multi-image/video token formatting, prefer qwen_vl_utils'
# process_vision_info helper as shown on the model card - the direct-image
# call below is a simplified but workable version for this demo.
try:
    vlm_model, vlm_processor = load_vlm(VLM_MODEL_ID)
    print('Loaded', VLM_MODEL_ID)
except Exception as e:
    print(f'{VLM_MODEL_ID} did not fit/load ({e}); falling back to {VLM_FALLBACK_ID}')
    vlm_model, vlm_processor = load_vlm(VLM_FALLBACK_ID)

In [ ]:
def generate_commentary(clip_frames, event_context, max_new_tokens=80):
    # clip_frames: list of PIL images (a few fps sample of a ~5-10s clip)
    # event_context: the structured facts already computed by tracking + Jersey-CNN,
    # passed in as text so the VLM narrates measured events instead of hallucinating them.
    context_str = json.dumps(event_context)
    messages = [{
        'role': 'user',
        'content': [
            *[{'type': 'image', 'image': f} for f in clip_frames],
            {'type': 'text', 'text': (
                'You are a soccer broadcast commentator. Using the tracked events below, '
                f'give a short, natural 1-2 sentence commentary. Tracked events: {context_str}'
            )},
        ],
    }]
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vlm_processor(text=[text], images=clip_frames, return_tensors='pt').to(vlm_model.device)
    with torch.no_grad():
        out_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return vlm_processor.batch_decode(out_ids[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]

In [ ]:
def build_tactical_report(event_log, jersey_map, template_only=True):
    # Always-available template summary (no GPU needed) so the demo never
    # fails to produce a report even if the VLM step is skipped for time.
    touches = {}
    for e in event_log:
        if e['type'] == 'touch':
            touches[e['track_id']] = touches.get(e['track_id'], 0) + 1
    lines = ['Match Report', '------------']
    for track_id, count in sorted(touches.items(), key=lambda kv: -kv[1]):
        number = jersey_map.get(track_id, '?')
        lines.append(f'#{number}: {count} touches')
    report_text = '\n'.join(lines)

    if not template_only:
        prompt = f'Turn this raw stat sheet into a two-paragraph coach-facing tactical summary:\n{report_text}'
        inputs = vlm_processor(text=[prompt], return_tensors='pt').to(vlm_model.device)
        with torch.no_grad():
            out_ids = vlm_model.generate(**inputs, max_new_tokens=180)
        report_text = vlm_processor.batch_decode(out_ids[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]

    return report_text

In [ ]:
RUN_OPTIONAL_LORA = False   # flip to True only if phases 1-5 finished comfortably inside budget

if RUN_OPTIONAL_LORA:
    from peft import LoraConfig, get_peft_model

    lora_cfg = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj'],
        task_type='CAUSAL_LM',
    )
    vlm_model = get_peft_model(vlm_model, lora_cfg)
    vlm_model.print_trainable_parameters()

    # Point this at a small local subset of SoccerChat you've converted to
    # JSONL per the dataset's own instructions (requires the SoccerNet NDA
    # for video access). Keep it to a few hundred clips - this is a light
    # domain-adaptation pass, not the full 85k-clip recipe.
    lora_tb = TimeBudget(CFG['vlm_time_budget_hours'] * 0.5)
    print('Optional LoRA pass would run here - wire in your SoccerChat JSONL subset and a standard HF Trainer loop.')
    log_phase('vlm_lora_optional', lora_tb)
else:
    print('Skipping optional LoRA fine-tune - using the pretrained SoccerChat checkpoint as-is.')

### Phase 6 — End-to-end demo

Wires every trained + classical component together and runs on a sample clip. If you have not attached a real amateur match clip yet, a synthetic smoke-test clip is generated automatically so the full pipeline is provably runnable end to end.

In [ ]:
def run_halo_on_video(video_path, output_path, max_frames=300, sample_every=1):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    roi_engine = DynamicROIEngine()
    track_history, ball_history = {}, []
    jersey_map = {}
    sample_frames_for_vlm = []

    frame_idx = 0
    while cap.isOpened() and frame_idx < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % sample_every != 0:
            frame_idx += 1
            continue

        results = yolo_model.predict(frame, verbose=False, device=0 if torch.cuda.is_available() else 'cpu')[0]
        detections = sv.Detections.from_ultralytics(results)
        tracked = tracker.update_with_detections(detections)

        player_boxes = [box for box, cls in zip(tracked.xyxy, tracked.class_id) if CLASS_NAMES[cls] in ('player', 'goalkeeper')]
        ball_boxes = [box for box, cls in zip(tracked.xyxy, tracked.class_id) if CLASS_NAMES[cls] == 'ball']

        avg_player_h = np.mean([b[3] - b[1] for b in player_boxes]) if player_boxes else 80.0
        ball_center = None
        if ball_boxes:
            bx1, by1, bx2, by2 = ball_boxes[0]
            ball_center = ((bx1 + bx2) / 2, (by1 + by2) / 2)
        roi = roi_engine.update(ball_center, avg_player_h)
        ball_history.append((frame_idx, *roi['center']))

        for box, tid in zip(tracked.xyxy, tracked.tracker_id):
            x1, y1, x2, y2 = box
            track_history.setdefault(tid, []).append((frame_idx, (x1 + x2) / 2, (y1 + y2) / 2))
            torso = frame[int(y1):int(y1 + (y2 - y1) * 0.4), int(x1):int(x2)]
            if torso.size > 0 and tid not in jersey_map:
                crop = cv2.resize(torso, (JERSEY_IMG_SIZE, JERSEY_IMG_SIZE))
                crop_t = torch.from_numpy(crop / 127.5 - 1.0).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    p_vis, p_tens, p_units = jersey_model(crop_t)
                if p_vis.argmax().item() == 1:
                    jersey_map[tid] = f'{p_tens.argmax().item()}{p_units.argmax().item()}'

            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            label = f"#{jersey_map.get(tid, '?')} id{tid}"
            cv2.putText(frame, label, (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        cx, cy = roi['center']
        cv2.circle(frame, (int(cx), int(cy)), int(roi['radius_px']), (0, 165, 255), 2)

        if len(sample_frames_for_vlm) < 8 and frame_idx % max(1, max_frames // 8) == 0:
            sample_frames_for_vlm.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()

    events = detect_touches_and_passes(track_history, ball_history, pitch_scale_m_per_px=0.02)
    event_context = {
        'total_events': len(events),
        'touches_per_player': {jersey_map.get(tid, str(tid)): sum(1 for e in events if e.get('track_id') == tid) for tid in track_history},
    }

    commentary = 'skipped (no VLM loaded)'
    if 'vlm_model' in globals() and sample_frames_for_vlm:
        try:
            commentary = generate_commentary(sample_frames_for_vlm, event_context)
        except Exception as e:
            commentary = f'VLM commentary failed: {e}'

    report = build_tactical_report(events, jersey_map, template_only=True)

    return {'annotated_video': str(output_path), 'events': events, 'commentary': commentary, 'report': report}

In [ ]:
DEMO_DIR = OUTPUT_DIR / 'demo'
sample_video_path = DEMO_DIR / 'sample_input.mp4'

def make_synthetic_demo_clip(path, n_frames=150, size=(640, 360)):
    # Smoke-test clip so the pipeline is provably runnable even with no real
    # footage attached yet - two colored dots (ball + player) moving on a green pitch.
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), 25, size)
    for i in range(n_frames):
        frame = np.full((size[1], size[0], 3), (34, 120, 34), dtype=np.uint8)
        px = int(size[0] / 2 + 150 * math.sin(i / 20))
        py = int(size[1] / 2 + 60 * math.cos(i / 15))
        bx = int(px + 30 * math.sin(i / 6))
        by = int(py + 15 * math.cos(i / 6))
        cv2.circle(frame, (px, py), 12, (0, 0, 200), -1)
        cv2.circle(frame, (bx, by), 6, (255, 255, 255), -1)
        writer.write(frame)
    writer.release()

if not sample_video_path.exists():
    # If you've attached your own clip via Kaggle "Add Data", point
    # sample_video_path at it instead of generating the synthetic one.
    make_synthetic_demo_clip(sample_video_path)
    print('No real demo clip found - generated a synthetic smoke-test clip instead.')

demo_result = run_halo_on_video(
    sample_video_path,
    DEMO_DIR / 'annotated_output.mp4',
    max_frames=150,
)
print('Commentary:', demo_result['commentary'])
print(demo_result['report'])

### Phase 7 — Package everything for download

In [ ]:
(OUTPUT_DIR / 'logs' / 'run_config.json').write_text(json.dumps(CFG, indent=2))

zip_path = shutil.make_archive(str(OUTPUT_DIR.parent / 'halo_run_package'), 'zip', root_dir=OUTPUT_DIR)
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Packaged everything into: {zip_path} ({size_mb:.1f} MB)')
print('Download it from the Kaggle "Output" tab of this notebook session.')

In [ ]:
total_elapsed = sum(PHASE_LOG.values())
print('Phase-by-phase wall clock:')
for name, secs in PHASE_LOG.items():
    print(f'  {name:20s}: {secs/60:6.1f} min')
print(f'TOTAL: {total_elapsed/3600:.2f} h (target 6-8h on a single Tesla T4)')